In the work, the authors considered the classical Heisenberg model on the 2D square lattice.
The local magnetic moment $\mathbf{M}$ is the local spin $\mathbf{S}$ in JuMag.

!!! note "Used parameters in the simulation"
    |Parameter | Value  |
    | :----:   | :----: |
    | Lattice constant | $a = 0.5$ nm |
    | Spin length      | $S = 1$      |
    | Magnetic moment  |  $\mu_s = 2 \mu_B$ |
    | Excahnge constant |  $J = 1$ meV   |
    | DMI         | $D/J = 0.18$  |     |
    | External field  | $H \mu_s /J  = 0.015$ |
    | Spin-polarization rate        |  $p = 0.2$   |

In [1]:
using JuMag
using Printf
using NPZ

JuMag.cuda_using_double(true)

function m0_fun(i, j, k, dx, dy, dz)
    r = 25
    if ((i - 80)^2 + (j - 40)^2 < r^2)
        return (0.05, 0.01, -1)
    end
    return (0,0,1)
end

function relax_system()
    J = 1*meV
    D = 0.18*J

    mesh = CubicMeshGPU(nx=166, ny=80, nz=1, dx=0.5e-9, dy=0.5e-9, pbc="xy")

    sim = Sim(mesh, driver="SD", name="skx")
    set_mu_s(sim, mu_s_1)

    add_exch(sim, J, name="exch")
    add_dmi(sim, D, name="dmi")

    Hz= 1.5e-2*J / mu_s_1
    add_zeeman(sim, (0,0,Hz))
    init_m0(sim, m0_fun)

    relax(sim, maxsteps=1000, stopping_dmdt=1e-5, using_time_factor=false)

    save_vtk(sim, "skx")

    Q = compute_skyrmion_number(Array(sim.spin),mesh)

    return Q
end

relax_system()

[ Info: Running Driver : JuMag.EnergyMinimizationGPU{Float64}.
[ Info: step =    1  step_size=1.000000e-10    max_dmdt=5.149916e+00
[ Info: step =    2  step_size=8.436987e-02    max_dmdt=5.032525e+00
[ Info: step =    3  step_size=2.258153e-02    max_dmdt=1.089018e+01
[ Info: step =    4  step_size=2.189487e-01    max_dmdt=7.299800e+00
[ Info: step =    5  step_size=4.665599e-02    max_dmdt=2.641813e+01
[ Info: step =    6  step_size=2.926695e-02    max_dmdt=2.392545e+01
[ Info: step =    7  step_size=1.731259e-02    max_dmdt=1.792657e+01
[ Info: step =    8  step_size=1.678681e-02    max_dmdt=2.928036e+00
[ Info: step =    9  step_size=1.802116e-02    max_dmdt=1.773553e+00
[ Info: step =   10  step_size=2.987386e-01    max_dmdt=1.566169e+00
[ Info: step =   11  step_size=1.568115e-01    max_dmdt=3.710826e+00
[ Info: step =   12  step_size=6.373114e-02    max_dmdt=1.045289e+01
[ Info: step =   13  step_size=1.564975e-02    max_dmdt=1.775098e+01
[ Info: step =   14  step_size=1.498179e

-1.0

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*